In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import convert_dtypes

In [3]:
seller_features = pd.read_csv(r"C:\Users\Lenovo\Customer-Intelligence-Platform\data\processed\seller_summary_clean.csv")

In [4]:
seller_features.head(5)

,seller_id,seller_city,seller_state,total_orders,total_units_sold,total_revenue,total_freight,avg_product_price,avg_review_score,avg_delivery_days
0,0015a82c2db000af6aaaf3ae2ecb0532,santo andre,SP,3,3,2685.00,63.06,895.00,3.67,10.67
1,001cca7ae9ae17fb1caed9dfb1094831,cariacica,ES,200,239,25080.03,8854.14,104.94,3.90,13.11
2,001e6ad469a905060d959994f1b41e4f,sao goncalo,RJ,1,1,250.00,17.94,250.00,1.00,NaN
3,002100f778ceb8431b7a1020ff7ab48f,franca,SP,51,55,1234.50,793.66,22.45,3.98,16.00
4,003554e2dce176b5555353e4f3555ac8,goiania,GO,1,1,120.00,19.38,120.00,5.00,4.00


### Feature 1: Revenue per order
Measures the average revenue generated by a seller from each completed order.

In [6]:
seller_features["revenue_per_order"] = (
    seller_features["total_revenue"]
    / seller_features["total_orders"]
)

In [8]:
seller_features["revenue_per_order"].describe()


count    3095.000000
mean      194.653787
std       346.490015
min         3.500000
25%        60.124693
50%       105.870000
75%       189.970833
max      6729.000000
Name: revenue_per_order, dtype: float64

In [10]:
seller_features[
    [
        "total_revenue",
        "total_orders",
        "revenue_per_order"
    ]
].head()

,total_revenue,total_orders,revenue_per_order
0,2685.00,3,895.000000
1,25080.03,200,125.400150
2,250.00,1,250.000000
3,1234.50,51,24.205882
4,120.00,1,120.000000


### Feature 2: Freight Ratio
Measures shipping cost relative to seller revenue.

In [12]:
seller_features["freight_ratio"] = (
    seller_features["total_freight"]
    / seller_features["total_revenue"]
)

In [13]:
seller_features["freight_ratio"].describe()

count    3095.000000
mean        0.265439
std         0.231639
min         0.011905
25%         0.125681
50%         0.205172
75%         0.327371
max         3.535000
Name: freight_ratio, dtype: float64

### Feature 3: Premium Seller
Identifies sellers specializing in higher-priced products.

In [14]:
premium_threshold = seller_features["avg_product_price"].quantile(0.80)

seller_features["premium_seller"] = (
seller_features["avg_product_price"] >= premium_threshold
).astype(int)

print("Premium seller threshold:", premium_threshold)

seller_features["premium_seller"].value_counts()

Premium seller threshold: 202.97400000000005


premium_seller
0    2476
1     619
Name: count, dtype: int64

### Feature 4: High Rating Seller

In [15]:
seller_features["high_rating_seller"] = (
    seller_features["avg_review_score"] >= 4.5
).astype(int)

seller_features["high_rating_seller"].value_counts(dropna=False)

high_rating_seller
0    2108
1     987
Name: count, dtype: int64

### Feature 5: Fast Delivery seller

In [17]:
delivery_threshold = seller_features["avg_delivery_days"].median()

seller_features["fast_delivery_seller"] = (
    seller_features["avg_delivery_days"] <= delivery_threshold
).astype(int)

print("Median delivery days:", delivery_threshold)

seller_features["fast_delivery_seller"].value_counts()

Median delivery days: 11.0


fast_delivery_seller
0    1595
1    1500
Name: count, dtype: int64